# M3-N 관계의 CLV 구성성향별 무학습 진단

기존 Dunnhumby seed 42 M3-N과 M1 체크포인트만 불러와 N-성향($\pi_N>0.5$) 고객에서 Revenue@10과 Recall@20이 개선됐는지 확인합니다. 체크포인트가 없으면 새로 학습하지 않고 중단합니다. test와 holdout은 평가하지 않습니다.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
import shutil, subprocess

REVIEWED_SHA = '4322476cb2d51ace30266b21dee1ddb62f030a84'
repo = Path('/content/clv-m2-lightgcn-runner')
if repo.exists():
    shutil.rmtree(repo)
subprocess.run(['git', 'clone', '-q', 'https://github.com/jung-un/clv-m2-lightgcn-runner.git', str(repo)], check=True)
subprocess.run(['git', '-C', str(repo), 'checkout', '-q', REVIEWED_SHA], check=True)
actual_sha = subprocess.check_output(['git', '-C', str(repo), 'rev-parse', 'HEAD'], text=True).strip()
assert actual_sha == REVIEWED_SHA, (actual_sha, REVIEWED_SHA)
%cd /content/clv-m2-lightgcn-runner
print('검토 코드 고정 완료:', actual_sha)

In [ ]:
import json, torch
from lightgcn_clv_m3_transfer import configure_m3_transfer_dunnhumby_run
from lightgcn_clv_m3_n_segment_diagnostic import run_diagnostic

cfg = configure_m3_transfer_dunnhumby_run()
assert torch.cuda.is_available(), '런타임 > 런타임 유형 변경에서 GPU를 선택하세요.'
print(json.dumps({
    'dataset': cfg['DATASET'],
    'seed': cfg['SEED_LIST'][0],
    'split': 'validation only',
    'training': False,
    'checkpoint_policy': 'existing M1 and M3-N only; fail closed if absent',
    'pi_N': 'log(1+N_u) / (log(1+N_u)+log(1+V_u))',
}, ensure_ascii=False, indent=2))

In [ ]:
diagnostic_df = run_diagnostic(cfg)

In [ ]:
from IPython.display import display

display(diagnostic_df)
print('본실행 진행 판정:')
print(json.dumps(diagnostic_df.attrs['decision'], ensure_ascii=False, indent=2))
print('결과 파일:', diagnostic_df.attrs['paths'])